## Lab 1: Chunking baseline vs improved

Goal: use fixed-size chunking as a baseline and measure how sentence-aware and semantic chunking change retrieval quality on the same corpus. This is not about a single winner, but about tradeoffs and improvements vs baseline.

In [43]:
import sys
sys.path.append('Track2/Functions')

from pathlib import Path

from C2_1_Func import (
    build_corpus_documents, persist_corpus, load_eval_queries, load_corpus,
    tokenize, SimpleTfidfVectorizer, split_sentences, fixed_size_chunking,
    sentence_aware_chunking, cosine_similarity, semantic_chunking,
    build_chunks, cosine_similarity_scores, retrieve_top_k, evaluate_chunking,
    BM25, bm25_retrieve,
)

DATA_DIR = Path('Track2') / 'demos' / 'data' / 'C2.1_corpus'
documents = build_corpus_documents()
persist_corpus(documents, DATA_DIR)
corpus_queries = load_eval_queries(DATA_DIR)
ingested_docs = load_corpus(DATA_DIR)

In [ ]:

from pathlib import Path
from C2_1_Func import build_comparison_corpus, persist_comparison_corpus, load_comparison_corpus

# ── Comparison corpus: one multi-topic doc per domain pair (persisted to disk) ──
COMPARISON_CORPUS_DIR = Path('Track2') / 'Labs' / 'data' / 'LabC2.1_corpus'
_raw_comparison_docs, _raw_comparison_queries = build_comparison_corpus()
persist_comparison_corpus(_raw_comparison_docs, _raw_comparison_queries, COMPARISON_CORPUS_DIR)
comparison_docs, comparison_queries = load_comparison_corpus(COMPARISON_CORPUS_DIR)

fixed_chunker_eval   = lambda text: fixed_size_chunking(text, chunk_size=10, overlap=0)
sentence_chunker_eval = lambda text: sentence_aware_chunking(text, max_words=25, overlap_sentences=1)
semantic_chunker_eval = lambda text: semantic_chunking(text, similarity_threshold=0.45, min_sentences=1, max_words=60)

baseline = evaluate_chunking('Fixed-size (baseline)', fixed_chunker_eval,   comparison_docs, comparison_queries)
sentence = evaluate_chunking('Sentence-aware',        sentence_chunker_eval, comparison_docs, comparison_queries)
semantic = evaluate_chunking('Semantic',              semantic_chunker_eval, comparison_docs, comparison_queries)

results = [baseline, sentence, semantic]

print('Chunking comparison — domain corpus (finance, education, healthcare, technology)')
print(f'{"Method":<22} | Top1 | Top3 |  MRR | Delta vs baseline')
for r in results:
    if r is baseline:
        delta = '---'
    else:
        delta = (f"{r['top1']-baseline['top1']:+.2f}/"
                 f"{r['top3']-baseline['top3']:+.2f}/"
                 f"{r['mrr']-baseline['mrr']:+.2f}")
    print(f"{r['method']:<22} | {r['top1']:.2f} | {r['top3']:.2f} | {r['mrr']:.2f} | {delta}")

print()
print(f'{"Method":<22} | Chunks | Avg words')
for r in results:
    print(f"{r['method']:<22} | {r['chunks']:>6} | {r['avg_len']:>9.1f}")

Relative to the fixed-size baseline, sentence-aware improves Top1/Top3/MRR by +0.33/+0.33/+0.33, while semantic improves Top1 and MRR but keeps Top3 flat. The tradeoff snapshot shows sentence-aware yields fewer, larger chunks, while semantic yields more, smaller chunks.

## Lab 2: RAG pipeline build

Build a full pipeline: ingest -> chunk -> embed -> retrieve -> grounded generate. Then compare chunk size variants on retrieval quality.

## Hallucination reduction through grounding

Without retrieval, a generative system must rely entirely on parametric memory. For facts that were not seen during training — or that have changed since — it fills the gap with confident-sounding but wrong text. This is called hallucination.

RAG prevents this by **grounding**: the answer must be assembled from retrieved chunks. If the evidence is absent or contradicts the model, the model either acknowledges the gap or stays silent rather than fabricating an answer.

The demo below simulates both modes:
- **Ungrounded** — the system picks the most query-relevant single word from the index, mimicking a model with only weak memory signals and no context.
- **Grounded** — the system builds the answer sentence-by-sentence strictly from the top retrieved chunks.

In a real LLM-backed system the gap is even larger: ungrounded answers look fluent and confident, which makes hallucinations harder to detect.

In [ ]:
def ungrounded_answer(query):
    """
    Simulates a model answering from parametric memory only:
    picks the single highest-scoring term from the full corpus without
    returning any supporting text. Produces incomplete, decontextualised answers.
    """
    query_terms = set(tokenize(query))
    best_term   = None
    best_score  = 0.0
    for doc in ingested_docs:
        for term in tokenize(doc['body']):
            if term in query_terms:
                score = len(term)   # favour longer matching terms
                if score > best_score:
                    best_score = score
                    best_term  = term
    return best_term or '[no answer found]'


def grounded_answer(query, retrieve_fn, top_k=3):
    """
    Assembles an answer only from sentences inside retrieved chunks.
    If no retrieved chunk contains query terms, returns a refusal rather
    than fabricating a response.
    """
    retrieved = retrieve_fn(query, top_k)
    query_terms = set(tokenize(query))
    candidates = []
    for chunk in retrieved:
        for sentence in split_sentences(chunk['text']):
            score = sum(1 for t in tokenize(sentence) if t in query_terms)
            if score > 0:
                candidates.append((score, sentence, chunk['title']))
    if not candidates:
        return '[No supporting evidence found in the corpus - cannot answer.]'
    candidates.sort(key=lambda x: x[0], reverse=True)
    top = candidates[:2]
    answer_text = ' '.join(s for _, s, _ in top)
    source_docs = list(dict.fromkeys(title for _, _, title in top))
    return f'{answer_text}  [sources: {", ".join(source_docs)}]'


# Build a BM25 index for the hallucination demo below
hallu_chunker = lambda text: sentence_aware_chunking(text, max_words=50, overlap_sentences=1)
hallu_chunks  = build_chunks(ingested_docs, hallu_chunker)
hallu_bm25    = BM25().fit([c['text'] for c in hallu_chunks])
hallu_bm25_retrieve = lambda query, top_k=3: bm25_retrieve(query, hallu_bm25, hallu_chunks, top_k=top_k)

# ── Hallucination demo on cross-domain questions ──────────────────────────────
hallucination_queries = [
    {
        'domain':   'Finance',
        'question': 'Why do central banks raise interest rates?',
        'expected': 'control inflation',
    },
    {
        'domain':   'Education',
        'question': 'What does the Zone of Proximal Development describe?',
        'expected': 'gap between what a learner can do alone and with guidance',
    },
    {
        'domain':   'Healthcare',
        'question': 'What is the gold standard for evaluating medical treatments?',
        'expected': 'randomized controlled trials',
    },
    {
        'domain':   'Technology',
        'question': 'What do containers package for portability?',
        'expected': 'application and its dependencies',
    },
    {
        'domain':   'Legal',
        'question': 'What rights do individuals have under GDPR?',
        'expected': 'access, correct, and delete their data',
    },
]

print('Hallucination reduction demo')
print('=' * 90)
for item in hallucination_queries:
    ug = ungrounded_answer(item['question'])
    gr = grounded_answer(item['question'], hallu_bm25_retrieve, top_k=3)
    correct_tag = 'CORRECT' if item['expected'].split()[0].lower() in gr.lower() else 'WRONG'
    print(f"Domain   : {item['domain']}")
    print(f"Query    : {item['question']}")
    print(f"Expected : {item['expected']}")
    print(f"Ungrounded (no context) -> \"{ug}\"  <- incomplete / potentially hallucinated")
    print(f"Grounded  (RAG)         -> {gr}  {correct_tag}")
    print('-' * 90)

print()
print('Key observation: the grounded answer cites the source document and quotes')
print('verbatim evidence. The ungrounded answer has no supporting context and')
print('would be trivially wrong for out-of-distribution or recently-changed facts.')

In [ ]:
def build_rag_index(docs, chunk_fn):
    chunks = build_chunks(docs, chunk_fn)
    vectorizer = SimpleTfidfVectorizer()
    vectors = vectorizer.fit_transform([c['text'] for c in chunks])
    return chunks, vectorizer, vectors

def generate_answer(query, retrieved_chunks):
    query_terms = set(tokenize(query))
    candidates = []
    for chunk in retrieved_chunks:
        for sentence in split_sentences(chunk['text']):
            score = sum(1 for t in tokenize(sentence) if t in query_terms)
            if score > 0:
                candidates.append((score, sentence))
    if not candidates:
        return retrieved_chunks[0]['text'] if retrieved_chunks else ''
    candidates.sort(key=lambda x: x[0], reverse=True)
    top_sentences = [s for _, s in candidates[:2]]
    return ' '.join(top_sentences)

In [ ]:
small_chunker = lambda text: fixed_size_chunking(text, chunk_size=8, overlap=0)
large_chunker = lambda text: fixed_size_chunking(text, chunk_size=40, overlap=5)

small = evaluate_chunking('Fixed-8', small_chunker, ingested_docs, corpus_queries)
large = evaluate_chunking('Fixed-40', large_chunker, ingested_docs, corpus_queries)

print('Chunk size comparison (same method, different sizes)')
print('Method | Chunks | Avg words | Top1 | Top3 | MRR')
for r in [small, large]:
    print(f"{r['method']:<8} | {r['chunks']:>6} | {r['avg_len']:>9.1f} | {r['top1']:.2f} | {r['top3']:.2f} | {r['mrr']:.2f}")

Chunk size comparison (same method, different sizes)
Method | Chunks | Avg words | Top1 | Top3 | MRR
Fixed-8  |    118 |       6.8 | 0.58 | 0.83 | 0.71
Fixed-40 |     60 |      13.4 | 1.00 | 1.00 | 1.00


Using the smaller fixed-size chunks as the baseline, the larger chunk size improves Top1/Top3/MRR and restores context. This reinforces that chunk size tuning is a baseline-driven improvement, not a universal winner.

In [ ]:

# ── Full RAG pipeline: end-to-end demo on domain-specific queries ─────────────
# Build index with sentence-aware chunking (best all-round in Lab 1)
rag_chunker = lambda text: sentence_aware_chunking(text, max_words=50, overlap_sentences=1)
rag_chunks, rag_vectorizer, rag_vectors = build_rag_index(ingested_docs, rag_chunker)

domain_demo_queries = [
    {'domain': 'Finance',    'query': 'Why do central banks raise interest rates?'},
    {'domain': 'Finance',    'query': 'What measures a bond sensitivity to rate changes?'},
    {'domain': 'Education',  'query': 'What does the Zone of Proximal Development describe?'},
    {'domain': 'Education',  'query': 'How does spaced repetition strengthen memory?'},
    {'domain': 'Healthcare', 'query': 'What is the gold standard for evaluating medical treatments?'},
    {'domain': 'Technology', 'query': 'What do containers package for portability?'},
    {'domain': 'Legal',      'query': 'What rights do individuals have under GDPR?'},
]

print('Full RAG pipeline — domain-specific query demo')
print('Pipeline: ingest → sentence-aware chunk → TF-IDF embed → top-3 retrieve → grounded generate')
print('=' * 90)

for item in domain_demo_queries:
    retrieved = retrieve_top_k(item['query'], rag_chunks, rag_vectorizer, rag_vectors, top_k=3)
    answer    = generate_answer(item['query'], retrieved)
    top_doc   = retrieved[0]['title'] if retrieved else 'n/a'
    print(f"[{item['domain']}]  {item['query']}")
    print(f"  → Top doc : {top_doc}")
    print(f"  → Answer  : {answer}")
    print()

The top chunk contains the exact evidence sentence, and the generated answer cites it directly, showing grounded generation from retrieved context.